In [1]:
%env DATA_PATH=../../../data
import requests
from lib.fit import load_fit_file, get_gps_data, get_camera_starts, get_camera_ends, get_sensor_data
import gpxpy
import pandas as pd
import glob
from tqdm.notebook import tqdm
import plotly.express as px
import plotly.graph_objects as go
import json
import os

DATA_PATH = '../../../data'


env: DATA_PATH=../../../data


In [2]:

url = "https://www.racebox.pro/webapp/sessions?type=track"

headers = {
    # "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:147.0) Gecko/20100101 Firefox/147.0",
    # "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    # "Accept-Language": "en-US,en;q=0.9",
    # "Accept-Encoding": "gzip, deflate, br, ?zstd",
    # "Connection": "keep-alive",
    # "Upgrade-Insecure-Requests": "1",
    # "Sec-Fetch-Dest": "document",
    # "Sec-Fetch-Mode": "navigate",
    # "Sec-Fetch-Site": "none",
    # "Sec-Fetch-User": "?1",
    # "Priority": "u=0, i",
    # "Pragma": "no-cache",
    # "Cache-Control": "no-cache",
}

#TODO: actual login logic
cookies = {
    "racebox": "6a03a63bbed584344d00986f",
    # "cookie-consent": "analytics%2Cmarketing%2Cessential",
    # "racebox-api-auth": "699d369f9a3322a136081c9b",
    # "analysis-map-mode": "google",
    # "analysis-bike-mode": "off",
    # "newsletter-subscribe-popup-dismissed": "true",
}

In [3]:
with requests.Session() as session:
    response = session.get(url, headers=headers, cookies=cookies)

    print("Status code:", response.status_code)

Status code: 200


In [6]:
from IPython.display import HTML

# HTML(response.text)

In [89]:
with requests.Session() as session:
    res = session.get("https://www.racebox.pro/webapp/session/69c91cf551d61c08910f9659/json", cookies=cookies)


In [90]:
raw = res.json()

In [91]:
import pandas as pd
import plotly.express as px
import numpy as np

In [92]:
columns = raw['session']['data']['dataColumns']
rows = raw['session']['data']['data']
df = pd.DataFrame(rows, columns=columns)

In [93]:
df['timestamp'] = df['iTOW'] - df['iTOW'].iloc[0]
df['Speed'] = df['Speed'] / 3.6  # kph to m/s
df.index = df['timestamp']

In [94]:
a_mag = pd.Series(np.linalg.norm(df[['GForceX', 'GForceY', 'GForceZ']], axis=1), index=df.index)

In [95]:
px.line(df.Speed)

In [96]:
fit = load_fit_file(f"{DATA_PATH}/virbs/all/2026-03-29-08-26-30.fit")

In [97]:
gps = get_gps_data(fit)

In [98]:
px.line(gps.speed)

In [17]:
from lib.signal import unfiorm_sample

In [99]:
fs = float(1000 / np.median(gps.timestamp.diff()[1:]))
fs

10.0

In [100]:
fit_speed = gps.speed.copy()
fit_speed.index = pd.to_timedelta(fit_speed.index * 1_000_000)
fit_speed = fit_speed.resample(f'{1/fs}s').mean()
fit_speed.values

array([2.47337826, 2.46989878, 2.46852182, ..., 0.01      , 0.02      ,
       0.01414214], shape=(1815,))

In [101]:
px.line(fit_speed)

In [102]:
rb_speed = df.Speed.loc[0:200_000].copy()
rb_speed.index = pd.to_timedelta(rb_speed.index * 1_000_000)
rb_speed = rb_speed.resample(f'{1/fs}s').mean()
rb_speed.values

array([1.96166667, 1.9045    , 2.032     , ..., 0.79166667, 0.7305    ,
       0.721     ], shape=(2001,))

In [103]:
px.line(rb_speed)

In [104]:
from scipy import signal

In [110]:
cor = signal.correlate(rb_speed - rb_speed.mean(), fit_speed - fit_speed.mean())
best_lag = signal.correlation_lags(len(rb_speed), len(fit_speed))[np.argmax(cor)]
best_lag

np.int64(54)

In [106]:
px.line(cor)

In [107]:
fit_speed_shifted = np.hstack([np.zeros(best_lag), fit_speed.values])
merged = pd.DataFrame(data=dict(
    fit=np.pad(fit_speed_shifted, (0, max(len(rb_speed) - len(fit_speed_shifted), 0))),
    rb=np.pad(rb_speed, (0, max(0, len(fit_speed_shifted) - len(rb_speed)))),
))

In [108]:
px.line(merged)

In [61]:
len(rb_speed)

2001

In [62]:
len(fit_speed_shifted) - len(rb_speed)

28

In [51]:

pd.DataFrame({
    'rb': rb_speed,
    'fit': fit_speed_shifted,
})

,rb,fit
timestamp,,
0 days 00:00:00,1.458333,NaN
0 days 00:00:00.100000,1.305500,NaN
0 days 00:00:00.200000,1.327333,NaN
0 days 00:00:00.300000,1.278500,NaN
0 days 00:00:00.400000,1.025000,NaN
...,...,...
0 days 00:04:18.831000,NaN,0.000000
0 days 00:04:18.931000,NaN,0.014142
0 days 00:04:19.031000,NaN,0.010000
